# Voice assistant on a free Colab GPU (self-managed instance)

Runs the whole stack on this Colab VM: faster-whisper (GPU), Gemma 3 via Ollama (GPU), Piper (CPU).
No hosted AI APIs. Run the cells top to bottom.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.

In [ ]:
# 1. Confirm a GPU is attached
%cd /content
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# 2. Get the project: clone from GitHub, or fall back to uploading voice-assistant.zip
import os, subprocess, zipfile
REPO_URL = "https://github.com/77Pratham/voice-assistant"   # change if your repo has another name
PROJECT = "/content/voice-assistant"
if not os.path.isdir(PROJECT):
    ok = bool(REPO_URL) and subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT]).returncode == 0
    if not ok:
        from google.colab import files
        name = next(iter(files.upload()))
        zipfile.ZipFile(name).extractall("/content")
%cd /content/voice-assistant


In [ ]:
# 3. Python dependencies (incl. the CUDA libraries faster-whisper needs), then check the GPU is visible
!pip install -q -r requirements.txt nvidia-cublas-cu12 nvidia-cudnn-cu12
import ctranslate2
print("CUDA devices seen by CTranslate2:", ctranslate2.get_cuda_device_count())

In [ ]:
# 4. Install and start Ollama, then pull the model
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, requests
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=1); break
    except Exception:
        time.sleep(1)
!ollama pull gemma3:4b

In [ ]:
# 5. Piper voices (Whisper large-v3-turbo downloads itself on first start)
!python scripts/download_models.py --no-whisper

In [ ]:
# 6. Start the assistant on the GPU and wait until models are loaded
import glob, os, site, subprocess, time, requests
cuda_libs = [p for sp in site.getsitepackages() for p in glob.glob(sp + "/nvidia/*/lib")]
env = {**os.environ, "HOST": "127.0.0.1", "PORT": "8000", "WHISPER_MODEL": "large-v3-turbo",
       "WHISPER_DEVICE": "cuda", "LLM_MODEL": "gemma3:4b", "SAVE_UTTERANCES": "1",
       "LD_LIBRARY_PATH": ":".join(cuda_libs + [os.environ.get("LD_LIBRARY_PATH", "")])}
server = subprocess.Popen(["python", "-m", "app"], env=env, stdout=open("/content/app.log", "w"), stderr=subprocess.STDOUT)
for _ in range(300):
    if server.poll() is not None:
        print(open("/content/app.log").read()[-3000:]); break
    try:
        print(requests.get("http://localhost:8000/api/health", timeout=2).json()); break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/app.log").read()[-3000:])

In [ ]:
# 6b. The LLM must say 100% GPU here. If it says CPU, re-run cell 4 and then cell 6.
!ollama ps


In [ ]:
# 7. Public HTTPS link (needed for the browser microphone). Open the printed URL on your laptop.
import re
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        print("Open this on your laptop:", m.group(0)); break
    time.sleep(1)

## Measure latency (for the README)
Run this once the assistant works. It writes `docs/BENCHMARK_RESULTS.md` with your hardware details.

In [ ]:
!python scripts/benchmark.py
print(open("docs/BENCHMARK_RESULTS.md").read())

In [ ]:
# Logs, if something looks wrong:
print(open("/content/app.log").read()[-3000:])

In [ ]:
# Debug speech recognition: download the clips the server actually received, and listen to them
!cd /content/voice-assistant/logs && zip -qr /content/utterances.zip utterances
from google.colab import files
files.download('/content/utterances.zip')


Colab sessions are temporary and idle ones disconnect, so start the tunnel and record your demo in one sitting.
Download `docs/BENCHMARK_RESULTS.md` before the session ends.